# Paralelna cjevovodna mreža: predvidi → izračunaj → provjeri

Tri grane spajaju ista dva čvora. Nepoznati su zajednički gubitak energije \(H\) i tri protoka. Faktor trenja ovisi o Reynoldsovu broju, pa sustav nije samo linearna podjela ukupnog protoka.

## Predvidi

1. Koja će grana preuzeti najveći protok: najkraća, najšira ili najglađa?
2. Moraju li protoci biti jednaki zato što je pad energije jednak?
3. Koja dva neovisna reziduala treba pratiti: čvorni i gransko-energetski?

Model koristi Darcy–Weisbachovu jednadžbu, lokalne gubitke i glatki prijelaz između laminarne i turbulentne aproksimacije. To je didaktički mrežni model, ne zamjena za kalibraciju stvarnog sustava.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({"figure.dpi": 110, "font.size": 10})
g, nu = 9.81, 1.0e-6

branches = [
    {"name": "A", "L": 95.0, "D": 0.090, "eps": 0.00005, "K": 2.0},
    {"name": "B", "L": 70.0, "D": 0.075, "eps": 0.00015, "K": 3.5},
    {"name": "C", "L": 130.0, "D": 0.105, "eps": 0.00010, "K": 1.5},
]

def friction_factor(Re, rel_roughness):
    if Re <= 0:
        return 0.0
    laminar = 64/Re
    turbulent = 0.25/np.log10(rel_roughness/3.7 + 5.74/Re**0.9)**2
    if Re <= 2300:
        return laminar
    if Re >= 4000:
        return turbulent
    weight = (Re-2300)/(4000-2300)
    return (1-weight)*laminar + weight*turbulent

def branch_loss(Q, branch):
    if Q <= 0:
        return 0.0
    area = np.pi*branch["D"]**2/4
    velocity = Q/area
    Re = velocity*branch["D"]/nu
    lam = friction_factor(Re, branch["eps"]/branch["D"])
    return (lam*branch["L"]/branch["D"] + branch["K"])*velocity**2/(2*g)

def flow_for_head(head, branch, q_upper, iterations=70):
    lo, hi = 0.0, q_upper
    for _ in range(iterations):
        mid = 0.5*(lo+hi)
        if branch_loss(mid, branch) < head:
            lo = mid
        else:
            hi = mid
    return 0.5*(lo+hi)

def solve_network(Q_total, branch_data, tolerance=1e-11, max_iterations=100):
    low = 0.0
    high = max(branch_loss(Q_total, b) for b in branch_data)
    history = []
    for iteration in range(max_iterations):
        head = 0.5*(low+high)
        flows = np.array([flow_for_head(head, b, Q_total) for b in branch_data])
        residual = flows.sum()-Q_total
        history.append((iteration, head, residual))
        if abs(residual) < tolerance:
            return head, flows, np.asarray(history)
        if residual > 0:
            high = head
        else:
            low = head
    raise RuntimeError("Mrežni rješavač nije konvergirao.")

Q_total = 0.030
head, flows, history = solve_network(Q_total, branches)
losses = np.array([branch_loss(q, b) for q, b in zip(flows, branches)])
for b, q, h in zip(branches, flows, losses):
    print(f"Grana {b['name']}: Q={1e3*q:6.3f} L/s, h={h:.6f} m")
print(f"Čvorni rezidual = {flows.sum()-Q_total:.3e} m3/s; zajednički H = {head:.6f} m")


## Izračunaj: nelinearno rješenje i osjetljivost

Vanjska bisekcija mijenja zajednički \(H\) dok zbroj granskih protoka ne zatvori čvor. Unutarnja bisekcija za svaku granu invertira nelinearnu funkciju \(h_i(Q_i)\). Povijest vanjskog reziduala pokazuje stvarnu konvergenciju, a ne samo konačan broj.

Zatim mijenjamo promjer grane B. Za svaki novi promjer ponovno rješavamo cijelu mrežu; ostali protoci se također moraju prilagoditi.


In [ ]:
D2_values = np.linspace(0.065, 0.095, 31)
flow_sensitivity = []
for diameter in D2_values:
    modified = [dict(b) for b in branches]
    modified[1]["D"] = diameter
    _, q_modified, _ = solve_network(Q_total, modified)
    flow_sensitivity.append(q_modified)
flow_sensitivity = np.asarray(flow_sensitivity)

print(f"Promjena Q_B: {1e3*flow_sensitivity[0,1]:.3f} → {1e3*flow_sensitivity[-1,1]:.3f} L/s")
print(f"Broj vanjskih iteracija osnovnog slučaja: {len(history)}")


## Provjeri

Čvorna bilanca provjerava očuvanje mase. Raspon granskih gubitaka provjerava energijsku kompatibilnost. Monotonost \(Q_B(D_B)\) dodatna je fizikalna provjera osjetljivosti.


In [ ]:
assert abs(flows.sum()-Q_total) < 1e-10
assert np.ptp(losses) < 1e-10
assert abs(history[-1,2]) < 1e-10
assert np.all(np.diff(flow_sensitivity[:,1]) > 0)
assert np.allclose(flow_sensitivity.sum(axis=1), Q_total, atol=1e-10)

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].semilogy(history[:,0], np.maximum(np.abs(history[:,2]), 1e-16), "o-", color="#b43c35")
axes[0].set(xlabel="vanjska iteracija", ylabel=r"$|\sum Q_i-Q|$ (m³/s)", title="Rezidual nelinearne mreže")
for i, b in enumerate(branches):
    axes[1].plot(1e3*D2_values, 1e3*flow_sensitivity[:,i], label=f"Q_{b['name']}")
axes[1].set(xlabel="promjer grane B (mm)", ylabel="granski protok (L/s)", title="Preraspodjela cijele mreže")
axes[1].legend()
for ax in axes: ax.grid(True, which="both", ls=":", alpha=.45)
plt.tight_layout(); plt.show()


## Protumači

U stvarnoj mreži hrapavost, otvorenost ventila i svojstva fluida nisu savršeno poznati. Dodaj ±10 % lokalnog koeficijenta grane B i provjeri mijenja li ta nesigurnost odluku o potrebnom promjeru, a ne samo treću decimalu protoka.
